In [5]:
import warnings

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

In [6]:
#Langkah 1: Generate & Eksplorasi Dataset Transaksi

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2–5 produk
transaksi = []

for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk,
                                           n_item,
                                           replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print("Contoh transaksi:")
print(transaksi[:3])

print("\nJumlah transaksi:", len(transaksi))

Contoh transaksi:
[[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]

Jumlah transaksi: 50


In [7]:
#Langkah 2: One-Hot Encoding Transaksi

from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

te_ary = te.fit(transaksi).transform(transaksi)

df = pd.DataFrame(te_ary, columns=te.columns_)

display(df.head())

,Gula,Keju,Kopi,Mentega,Roti,Selai,Sereal,Susu,Teh,Telur
0,False,True,True,True,True,True,False,False,False,False
1,False,False,True,True,True,True,False,False,True,False
2,False,False,True,False,False,False,False,True,True,False
3,False,True,False,False,False,True,False,False,True,True
4,True,True,False,True,False,False,False,True,False,False


In [8]:
#Langkah 3: Cari Frequent Itemset dengan Apriori

from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df,
                   min_support=ms,
                   use_colnames=True)

    print(f"min_support={ms}: {len(freq)} itemset ditemukan")

# Gunakan min_support yang menghasilkan jumlah itemset wajar
freq_items = apriori(df,
                     min_support=0.1,
                     use_colnames=True)

freq_items = freq_items.sort_values(
    by="support",
    ascending=False
)

display(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan


,support,itemsets
5,0.52,(Selai)
8,0.46,(Teh)
3,0.42,(Mentega)
9,0.36,(Telur)
1,0.34,(Keju)
0,0.32,(Gula)
2,0.32,(Kopi)
4,0.32,(Roti)
7,0.32,(Susu)
36,0.24,"(Teh, Selai)"


In [9]:
#Langkah 4: Bentuk & Saring Aturan Asosiasi
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    freq_items,
    metric="confidence",
    min_threshold=0.3
)

rules = rules[rules["lift"] > 1]

rules = rules.sort_values(
    by="lift",
    ascending=False
)

display(
    rules[
        ["antecedents",
         "consequents",
         "support",
         "confidence",
         "lift"]
    ].head(10)
)

,antecedents,consequents,support,confidence,lift
39,(Telur),"(Teh, Keju)",0.12,0.333333,2.380952
35,"(Teh, Keju)",(Telur),0.12,0.857143,2.380952
65,(Kopi),"(Mentega, Selai)",0.10,0.312500,1.953125
62,"(Mentega, Selai)",(Kopi),0.10,0.625000,1.953125
57,"(Roti, Gula)",(Selai),0.10,1.000000,1.923077
19,(Sereal),(Mentega),0.14,0.777778,1.851852
18,(Mentega),(Sereal),0.14,0.333333,1.851852
36,"(Teh, Telur)",(Keju),0.12,0.600000,1.764706
38,(Keju),"(Teh, Telur)",0.12,0.352941,1.764706
64,"(Selai, Kopi)",(Mentega),0.10,0.714286,1.700680


Aturan dengan lift terbesar menunjukkan hubungan pembelian yang paling kuat.
Jika muncul aturan Roti → Selai, maka aturan tersebut masuk akal karena pada dataset memang disisipkan pola bahwa pelanggan yang membeli roti cenderung membeli selai.

In [11]:
#Langkah 5: Rekomender Sederhana dengan Content-Based Filtering

from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    "produk": produk,
    "kategori": ["Bakery",
                 "Bakery",
                 "Dairy",
                 "Bakery",
                 "Dairy",
                 "Dairy",
                 "Minuman",
                 "Bumbu",
                 "Minuman",
                 "Dairy"]
})

fitur = pd.get_dummies(katalog["kategori"])

sim_matrix = cosine_similarity(fitur)


def rekomendasi_serupa(nama_produk, top_n=3):

    idx = katalog.index[
        katalog["produk"] == nama_produk
    ][0]

    skor = list(enumerate(sim_matrix[idx]))

    skor = sorted(
        skor,
        key=lambda x: x[1],
        reverse=True
    )

    skor = [
        s for s in skor
        if s[0] != idx
    ][:top_n]

    return katalog.iloc[
        [i for i, _ in skor]
    ]["produk"].tolist()


print("Mirip dengan Roti:")
print(rekomendasi_serupa("Roti"))

Mirip dengan Roti:
['Selai', 'Sereal', 'Susu']


In [12]:
#Langkah 6: Bandingkan Kedua Pendekatan

produk_target = "Roti"

rules_terkait = rules[
    rules["antecedents"].apply(
        lambda x: produk_target in list(x)
    )
]

print("=== Rekomendasi dari Association Rules ===")

display(
    rules_terkait[
        ["antecedents",
         "consequents",
         "lift"]
    ]
)

print("\n=== Rekomendasi dari Content-Based ===")
print(rekomendasi_serupa(produk_target))

=== Rekomendasi dari Association Rules ===


,antecedents,consequents,lift
57,"(Roti, Gula)",(Selai),1.923077
60,(Roti),"(Gula, Selai)",1.562500
58,"(Roti, Selai)",(Gula),1.420455
2,(Roti),(Selai),1.322115
20,(Roti),(Mentega),1.041667



=== Rekomendasi dari Content-Based ===
['Selai', 'Sereal', 'Susu']


- Kedua pendekatan dapat menghasilkan rekomendasi yang serupa apabila pola pembelian pelanggan sesuai dengan karakteristik produk. Namun, rekomendasi yang diberikan juga dapat berbeda karena masing-masing metode menggunakan pendekatan yang berbeda. Association Rules memanfaatkan pola pembelian dari data transaksi, sedangkan Content-Based Filtering mengandalkan kemiripan atribut atau kategori produk.
- Association Rules lebih tepat digunakan ketika tersedia data riwayat transaksi yang memadai untuk menemukan pola pembelian. Sebaliknya, Content-Based Filtering lebih sesuai apabila informasi mengenai karakteristik produk lebih lengkap. Dalam praktiknya, kedua metode dapat digabungkan menjadi hybrid recommendation system agar mampu menghasilkan rekomendasi yang lebih akurat, relevan, dan sesuai dengan kebutuhan pengguna.